In [1]:
# Run this cell to set the working directory to the repo root.
from pathlib import Path
import os, sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "method.py").exists() and (p / "tools").is_dir())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Uses `Distribution`, `run_de`, and `fitness` in the file
from method import *

# Clean up the output
np.set_printoptions(precision=4, suppress=True)

# Random seed for reproducibility
SEED = 20241225

## Real data experiments with the AI-READY dataset
The dataset is available at https://fairhub.io/datasets/2 upon request (one should request CGM and clinical variables data)  

After download, run `tools/processing_aireadi.py` and see `data/AI_READI_processing_guide.md` for the expected raw-data layout and outputs.


In [3]:
from pathlib import Path

from tools.processing_aireadi import load_ai_readi_cohort

REPO_ROOT = Path.cwd()
filtered_data = load_ai_readi_cohort(REPO_ROOT)

In [4]:
## More processing
filtered_data = filtered_data.copy()

# Additional lipid summaries used in the analyses below
filtered_data['tg_hdl_c'] = filtered_data['triglycerides'] / filtered_data['hdl_c']

print("Number of subjects in the processed AI-READI cohort:", filtered_data.shape[0])
print("\nResulting HbA1c range:", filtered_data['hba1c'].min(), "--", filtered_data['hba1c'].max())
print("Number of individuals with HbA1c < 5.7:", sum(filtered_data['hba1c'] < 5.7), f"out of {filtered_data.shape[0]} subjects")

Number of subjects in the processed AI-READI cohort: 573

Resulting HbA1c range: 4.5 -- 7.4
Number of individuals with HbA1c < 5.7: 313 out of 573 subjects


### Thresholds determination using $L_2$ loss

In [5]:
data_class = Distribution(filtered_data['gl'], ran=(40, 400), M=200)

In [ ]:
# Selecting K=2 thresholds
best_cutoffs2, min_loss2 = run_de(data_class, K=2, loss="Loss2", seed=SEED)
print("Cutoffs:", best_cutoffs2[1:-1])
print("Obtained loss:", min_loss2)

Cutoffs: [ 95.7497 169.4792]
Obtained loss: 45.48338136113584


In [ ]:
# Selecting K=2 thresholds with W1
best_cutoffs2, min_loss2 = run_de(data_class, K=2, loss="Loss2", Wdist="W1", seed=SEED)
print("Cutoffs:", best_cutoffs2[1:-1])
print("Obtained loss:", min_loss2)

Cutoffs: [ 97.8309 161.6412]
Obtained loss: 36.08015591891338


In [ ]:
_ = data_class.Wdist_matrix()
print("L2 loss at Traditional cutoffs [70, 181]:", fitness([70, 181], data_class, loss="Loss2"))

L2 loss at Traditional cutoffs [70, 181]: 103.38257775032335


In [ ]:
# Selecting K=4 thresholds (1.2 mins)
best_cutoffs2, min_loss2 = run_de(data_class, K=4, loss="Loss2", seed=SEED)
print("Cutoffs:", best_cutoffs2[1:-1])
print("Obtained loss:", min_loss2)

Cutoffs: [ 89.1604 127.7855 171.9481 231.3521]
Obtained loss: 5.513500557944302


In [5]:
# Selecting K=4 thresholds (1.2 mins)
best_cutoffs2, min_loss2 = run_de(data_class, K=4, loss="Loss2", Wdist="W1", seed=SEED)
print("Cutoffs:", best_cutoffs2[1:-1])
print("Obtained loss:", min_loss2)

Cutoffs: [ 87.9227 120.6726 160.632  209.9623]
Obtained loss: 4.165163489494456


In [ ]:
_ = data_class.Wdist_matrix()
print("L2 loss at Traditional cutoffs [54, 70, 181, 251]:", fitness([54, 70, 181, 251], data_class, loss="Loss2"))

L2 loss at Traditional cutoffs [54, 70, 181, 251]: 121.09803495736796


## Analysis using data-driven TIR metrics 

In [4]:
from tools.downstream_tools import build_tir_modeling_data

threshold_sets = {
    'trad_k2': [70, 181],
    'dd_k2': [96, 170],
    'trad_k4': [54, 70, 181, 251],
    'dd_k4': [90, 128, 172, 232],
}

modeling_data_total, feature_map = build_tir_modeling_data(filtered_data, threshold_sets, drop_gl=True)

trad_k2 = feature_map['trad_k2']
dd_k2 = feature_map['dd_k2']
trad_k4 = feature_map['trad_k4']
dd_k4 = feature_map['dd_k4']

In [5]:
from scipy.stats import kendalltau
col_interest = ['hdl_c', 'ldl_c', 'total_c', 'triglycerides', 'tg_hdl_c', 'hba1c']

kendall_k2 = pd.DataFrame(columns=trad_k2 + dd_k2 + ['hba1c'], index=col_interest)

for col in col_interest:
    for tir in trad_k2 + dd_k2 + ['hba1c']:
        x = modeling_data_total[col].values
        y = modeling_data_total[tir].values
        tau, p_value = kendalltau(x, y)
        # symbol = '***' if p_value < 0.001 else '** ' if p_value < 0.01 else '*  ' if p_value < 0.05 else '   '
        symbol = r'$^\ddagger$' if p_value < 0.001 else r'$^\dagger$' if p_value < 0.01 else r'$^*$' if p_value < 0.05 else r'$\:\,$'
        kendall_k2.loc[col, tir] = f"{tau:.3f}{symbol}"
print("Kendall's tau between health variables and TIR metrics")
# print("* p<0.05, ** p<0.01, *** p<0.001")
print("* p<0.05, † p<0.01, ‡ p<0.001")
kendall_k2

Kendall's tau between health variables and TIR metrics
* p<0.05, † p<0.01, ‡ p<0.001


,TIR_40_69,TIR_70_180,TIR_181_400,TIR_40_95,TIR_96_169,TIR_170_400,hba1c
hdl_c,0.060$^*$,"0.012$\:\,$","-0.033$\:\,$",0.088$^\dagger$,"-0.027$\:\,$","-0.042$\:\,$",-0.102$^\ddagger$
ldl_c,"0.002$\:\,$","-0.005$\:\,$","-0.002$\:\,$","-0.013$\:\,$","-0.002$\:\,$","-0.006$\:\,$","-0.015$\:\,$"
total_c,"-0.000$\:\,$","-0.015$\:\,$","0.018$\:\,$","-0.017$\:\,$","0.005$\:\,$","0.013$\:\,$","-0.012$\:\,$"
triglycerides,-0.086$^\dagger$,"-0.024$\:\,$",0.079$^\dagger$,-0.113$^\ddagger$,0.058$^*$,0.083$^\dagger$,0.093$^\dagger$
tg_hdl_c,-0.091$^\dagger$,"-0.021$\:\,$",0.071$^*$,-0.119$^\ddagger$,0.057$^*$,0.079$^\dagger$,0.116$^\ddagger$
hba1c,-0.161$^\ddagger$,-0.221$^\ddagger$,0.291$^\ddagger$,-0.219$^\ddagger$,"-0.031$\:\,$",0.299$^\ddagger$,1.000$^\ddagger$


In [6]:
kendall_k4 = pd.DataFrame(columns=trad_k4 + dd_k4 + ['hba1c'], index=col_interest)

for col in col_interest:
    for tir in trad_k4 + dd_k4 + ['hba1c']:
        x = modeling_data_total[col].values
        y = modeling_data_total[tir].values
        tau, p_value = kendalltau(x, y)
        # symbol = '***' if p_value < 0.001 else '** ' if p_value < 0.01 else '*  ' if p_value < 0.05 else '   '
        symbol = r'$^\ddagger$' if p_value < 0.001 else r'$^\dagger$' if p_value < 0.01 else r'$^*$' if p_value < 0.05 else r'$\:\,$'
        kendall_k4.loc[col, tir] = f"{tau:.3f}{symbol}"
print("Kendall's tau between health variables and TIR metrics")
# print("* p<0.05, ** p<0.01, *** p<0.001")
print("* p<0.05, † p<0.01, ‡ p<0.001")
kendall_k4

Kendall's tau between health variables and TIR metrics
* p<0.05, † p<0.01, ‡ p<0.001


,TIR_40_53,TIR_54_69,TIR_70_180,TIR_181_250,TIR_251_400,TIR_40_89,TIR_90_127,TIR_128_171,TIR_172_231,TIR_232_400,hba1c
hdl_c,"0.061$\:\,$",0.063$^*$,"0.012$\:\,$","-0.035$\:\,$","0.036$\:\,$",0.081$^\dagger$,0.125$^\ddagger$,-0.143$^\ddagger$,"-0.045$\:\,$","0.020$\:\,$",-0.102$^\ddagger$
ldl_c,"0.030$\:\,$","-0.008$\:\,$","-0.005$\:\,$","-0.003$\:\,$","0.021$\:\,$","-0.011$\:\,$","0.001$\:\,$","0.003$\:\,$","-0.004$\:\,$","0.033$\:\,$","-0.015$\:\,$"
total_c,"0.041$\:\,$","-0.009$\:\,$","-0.015$\:\,$","0.018$\:\,$","0.038$\:\,$","-0.015$\:\,$","0.020$\:\,$","-0.006$\:\,$","0.013$\:\,$","0.059$\:\,$","-0.012$\:\,$"
triglycerides,"-0.026$\:\,$",-0.098$^\ddagger$,"-0.024$\:\,$",0.080$^\dagger$,"0.022$\:\,$",-0.107$^\ddagger$,-0.083$^\dagger$,0.130$^\ddagger$,0.084$^\dagger$,"0.056$\:\,$",0.093$^\dagger$
tg_hdl_c,"-0.042$\:\,$",-0.101$^\ddagger$,"-0.021$\:\,$",0.073$^\dagger$,"0.001$\:\,$",-0.112$^\ddagger$,-0.113$^\ddagger$,0.156$^\ddagger$,0.080$^\dagger$,"0.032$\:\,$",0.116$^\ddagger$
hba1c,-0.078$^*$,-0.166$^\ddagger$,-0.221$^\ddagger$,0.293$^\ddagger$,0.236$^\ddagger$,-0.207$^\ddagger$,-0.248$^\ddagger$,0.265$^\ddagger$,0.299$^\ddagger$,0.272$^\ddagger$,1.000$^\ddagger$


### Linear model comparison
For Clarke's observation-wise likelihood-ratio test for non-nested models, we use the R package `clarkeTest`, imported through `rpy2` library of Python
- Note: `AIP` stands for the log-transformed TG/HDL-C ratio

In [11]:
from statsmodels.formula.api import ols
from tools.r_tools import setup_r_environment
from tools.downstream_tools import clarke_test

# Setting up R environment: see the function for details and make sure R and required packages are installed
setup_r_environment()

responses = ['hdl_c', 'ldl_c', 'total_c', 'log_triglycerides', 'AIP']

Using existing R_HOME: C:\Program Files\R\R-4.4.1


In [7]:
lm_results_k2 = pd.DataFrame()
lm_models_k2 = {}

for response in responses:
    formula_trad = f"{response} ~ " + " + ".join(trad_k2[:-1])
    model_trad = ols(formula_trad, data=modeling_data_total).fit()

    formula_dd = f"{response} ~ " + " + ".join(dd_k2[:-1])
    model_dd = ols(formula_dd, data=modeling_data_total).fit()

    formula_a1c = f"{response} ~ hba1c"
    model_a1c = ols(formula_a1c, data=modeling_data_total).fit()

    # If non-significant, skip the iteration (this excludes ldl_c and total_c consistently)
    if model_dd.f_pvalue >= 0.05 and model_trad.f_pvalue >= 0.05:
        continue

    # R^2 / AIC deltas
    lm_results_k2.at[response, 'R^2 (data-driven)'] = f"{model_dd.rsquared:.3f}"
    lm_results_k2.at[response, 'R^2 (traditional)'] = f"{model_trad.rsquared:.3f}" 

    resp = response[3:] if response.startswith('np.') else response

    clarke_stat, p_value = clarke_test(modeling_data_total, resp, trad_k2[:-1], dd_k2[:-1])
    lm_results_k2.at[response, "Clarke stat (p)"] = f"{clarke_stat:.3f} ({p_value:.3f})"

    symbol = r'$^\ddagger$' if p_value < 0.001 else r'$^\dagger$' if p_value < 0.01 else r'$^*$' if p_value < 0.05 else r'$\:\,$'
    lm_results_k2.at[response, 'AIC (trad - dd)'] = f"{model_trad.aic - model_dd.aic:.1f}{symbol}"

    lm_models_k2[response] = (model_trad, model_dd)

print("Linear model comparison between K=2 traditional and data-driven TIR thresholds")
lm_results_k2

Linear model comparison between K=2 traditional and data-driven TIR thresholds


,R^2 (data-driven),R^2 (traditional),Clarke stat (p),AIC (trad - dd)
hdl_c,0.018,0.011,238.000 (0.000),4.2$^\ddagger$
log_triglycerides,0.049,0.025,236.000 (0.000),14.5$^\ddagger$
AIP,0.050,0.026,232.000 (0.000),14.3$^\ddagger$


In [9]:
lm_results_k4 = pd.DataFrame()
lm_models_k4 = {}

for response in responses:
    formula_trad = f"{response} ~ " + " + ".join(trad_k4[:-1])
    model_trad = ols(formula_trad, data=modeling_data_total).fit()

    formula_dd = f"{response} ~ " + " + ".join(dd_k4[:-1])
    model_dd = ols(formula_dd, data=modeling_data_total).fit()

    # If non-significant, skip the iteration (this excludes ldl_c and total_c consistently)
    if model_dd.f_pvalue >= 0.05 and model_trad.f_pvalue >= 0.05:
        continue

    # R^2 / AIC deltas
    lm_results_k4.at[response, 'R^2 (data-driven)'] = f"{model_dd.rsquared:.3f}"
    lm_results_k4.at[response, 'R^2 (traditional)'] = f"{model_trad.rsquared:.3f}"

    resp = response[3:] if response.startswith('np.') else response

    clarke_stat, p_value = clarke_test(modeling_data_total, resp, trad_k4[:-1], dd_k4[:-1])
    lm_results_k4.at[response, "Clarke stat (p)"] = f"{clarke_stat:.3f} ({p_value:.3f})"

    symbol = r'$^\ddagger$' if p_value < 0.001 else r'$^\dagger$' if p_value < 0.01 else r'$^*$' if p_value < 0.05 else r'$\:\,$'
    lm_results_k4.at[response, 'AIC (trad - dd)'] = f"{model_trad.aic - model_dd.aic:.1f}{symbol}"
    
    lm_models_k4[response] = (model_trad, model_dd)

print("Linear model comparison between K=4 traditional and data-driven TIR thresholds")
lm_results_k4

Linear model comparison between K=4 traditional and data-driven TIR thresholds


,R^2 (data-driven),R^2 (traditional),Clarke stat (p),AIC (trad - dd)
hdl_c,0.041,0.014,232.000 (0.000),16.2$^\ddagger$
log_triglycerides,0.052,0.040,261.000 (0.037),7.5$^*$
AIP,0.060,0.040,253.000 (0.006),11.8$^\dagger$


In [10]:
lm_results = pd.concat([lm_results_k2.drop(columns=['Clarke stat (p)']).add_suffix(' (K=2)'), lm_results_k4.drop(columns=['Clarke stat (p)']).add_suffix(' (K=4)')], axis=1)
lm_results

,R^2 (data-driven) (K=2),R^2 (traditional) (K=2),AIC (trad - dd) (K=2),R^2 (data-driven) (K=4),R^2 (traditional) (K=4),AIC (trad - dd) (K=4)
hdl_c,0.018,0.011,4.2$^\ddagger$,0.041,0.014,16.2$^\ddagger$
log_triglycerides,0.049,0.025,14.5$^\ddagger$,0.052,0.040,7.5$^*$
AIP,0.050,0.026,14.3$^\ddagger$,0.060,0.040,11.8$^\dagger$


### AI-READI Wasserstein Regression benchmark


In [9]:
from tools.r_tools import ensure_r_packages, setup_r_environment
from tools.processing_aireadi import format_response_name
from tools.downstream_tools import compute_quantile_matrix, fit_wr_scalar_model, r_squared, build_tir_modeling_data

DE_THRESHOLD_SETS = {"de_k4": [90, 128, 172, 232]}
responses = ["hdl_c", "log_triglycerides", "AIP"]
probs = np.linspace(0.0, 1.0, 101)

setup_r_environment()

ensure_r_packages(
    ["fdapace", "remotes", "WR"],
    github_packages={"WR": "yqgchen/WR"},
)

modeling_data, wr_feature_map = build_tir_modeling_data(filtered_data, DE_THRESHOLD_SETS)
qf_matrix = compute_quantile_matrix(filtered_data["gl"], probs)
de_k4 = wr_feature_map["de_k4"]

print("Modeling data shape:", modeling_data.shape)
print("Quantile matrix shape (n x q):", qf_matrix.shape)
print("WR predictor orientation passed to R will be q x n:", qf_matrix.T.shape)


Using existing R_HOME: C:\Program Files\R\R-4.4.1
Modeling data shape: (573, 17)
Quantile matrix shape (n x q): (573, 101)
WR predictor orientation passed to R will be q x n: (101, 573)


#### Compare DE K=4 and Wasserstein regression via $R^2$


In [12]:
rows = []

for response in responses:
    formula_de_k4 = f"{response} ~ " + " + ".join(de_k4[:-1])
    model_de_k4 = ols(formula_de_k4, data=modeling_data).fit()

    y = modeling_data[response].to_numpy(dtype=float)
    wr_fit = fit_wr_scalar_model(y, qf_matrix, probs)

    rows.append(
        {
            "Response": format_response_name(response),
            "R^2 (DE K=4)": model_de_k4.rsquared,
            "R^2 (WR)": r_squared(y, wr_fit["fitted"]),
        }
    )

comparison_df = pd.DataFrame(rows).set_index("Response")

print("R^2 comparison between DE K=4 and Wasserstein regression")
comparison_df.round(3)


R^2 comparison between DE K=4 and Wasserstein regression


,R^2 (DE K=4),R^2 (WR)
Response,,
HDL-C,0.041,0.041
ldl_c,0.001,0.001
total_c,0.005,0.000
TG,0.052,0.054
TG/HDL-C,0.060,0.067


While WR outperforms DE with K=4, the gain of WR is marginal

## Export Kendall Tau Tables to LaTeX

In [ ]:
import os

col_interest = ['hdl_c', 'ldl_c', 'total_c', 'triglycerides', 'tg_hdl_c']

# Build the combined table with nested structure
combined_data = []

# Add K=2 Traditional rows
for tir in trad_k2:
    row_data = {'K': r'$K=2$', 'Method': 'Traditional', 'TIR': tir}
    for col in col_interest:
        row_data[col] = kendall_k2.loc[col, tir]
    combined_data.append(row_data)

# Add K=2 DE rows
for tir in dd_k2:
    row_data = {'K': r'$K=2$', 'Method': 'DE', 'TIR': tir}
    for col in col_interest:
        row_data[col] = kendall_k2.loc[col, tir]
    combined_data.append(row_data)

# Add K=4 Traditional rows
for tir in trad_k4:
    row_data = {'K': r'$K=4$', 'Method': 'Traditional', 'TIR': tir}
    for col in col_interest:
        row_data[col] = kendall_k4.loc[col, tir]
    combined_data.append(row_data)

# Add K=4 DE rows
for tir in dd_k4:
    row_data = {'K': r'$K=4$', 'Method': 'DE', 'TIR': tir}
    for col in col_interest:
        row_data[col] = kendall_k4.loc[col, tir]
    combined_data.append(row_data)

# Create the combined dataframe
combined_kendall = pd.DataFrame(combined_data)

# Display the combined table
print("Combined Kendall's tau table with nested structure:")
combined_kendall

Combined Kendall's tau table with nested structure:


,K,Method,TIR,hdl_c,ldl_c,total_c,triglycerides,tg_hdl_c
0,$K=2$,Traditional,TIR_40_69,0.060$^*$,"0.002$\:\,$","-0.000$\:\,$",-0.086$^\dagger$,-0.091$^\dagger$
1,$K=2$,Traditional,TIR_70_180,"0.012$\:\,$","-0.005$\:\,$","-0.015$\:\,$","-0.024$\:\,$","-0.021$\:\,$"
2,$K=2$,Traditional,TIR_181_400,"-0.033$\:\,$","-0.002$\:\,$","0.018$\:\,$",0.079$^\dagger$,0.071$^*$
3,$K=2$,DE,TIR_40_95,0.088$^\dagger$,"-0.013$\:\,$","-0.017$\:\,$",-0.113$^\ddagger$,-0.119$^\ddagger$
4,$K=2$,DE,TIR_96_169,"-0.027$\:\,$","-0.002$\:\,$","0.005$\:\,$",0.058$^*$,0.057$^*$
5,$K=2$,DE,TIR_170_400,"-0.042$\:\,$","-0.006$\:\,$","0.013$\:\,$",0.083$^\dagger$,0.079$^\dagger$
6,$K=4$,Traditional,TIR_40_53,"0.061$\:\,$","0.030$\:\,$","0.041$\:\,$","-0.026$\:\,$","-0.042$\:\,$"
7,$K=4$,Traditional,TIR_54_69,0.063$^*$,"-0.008$\:\,$","-0.009$\:\,$",-0.098$^\ddagger$,-0.101$^\ddagger$
8,$K=4$,Traditional,TIR_70_180,"0.012$\:\,$","-0.005$\:\,$","-0.015$\:\,$","-0.024$\:\,$","-0.021$\:\,$"
9,$K=4$,Traditional,TIR_181_250,"-0.035$\:\,$","-0.003$\:\,$","0.018$\:\,$",0.080$^\dagger$,0.073$^\dagger$


In [ ]:
# Function to convert to LaTeX with nested multirow
def create_latex_table_with_nested_multirow(df):
    """
    Convert the combined Kendall tau dataframe to LaTeX booktabs format with nested multirow
    K value spans all rows for that K, Method type spans rows within each K
    """
    latex_lines = []
    
    # Begin table
    latex_lines.append(r'\begin{table}[t]')
    latex_lines.append(r'\centering')
    latex_lines.append(r"\caption{Kendall's $\tau$ correlation between TIR metrics and health variables, with symbols indicating significant $p$-values under the null hypothesis of $\tau = 0$.}")
    latex_lines.append(r'\label{tab:kendall_tau}')
    latex_lines.append(r'\footnotesize')
    
    # Column specification: K, Method, TIR, then the health variables
    num_cols = len(col_interest)
    latex_lines.append(r'\begin{tabular}{lll' + 'r' * num_cols + '}')
    latex_lines.append(r'\toprule')
    
    # Header row
    header = r'$K$ & Method & Range (mg/dL)'
    for col in col_interest:
        # Format column names nicely
        if col == 'hdl_c':
            col_formatted = 'HDL-C'
        elif col == 'ldl_c':
            col_formatted = 'LDL-C'
        elif col == 'total_c':
            col_formatted = 'Total-C'
        elif col == 'triglycerides':
            col_formatted = 'TG'
        elif col == 'tg_hdl_c':
            col_formatted = 'TG/HDL-C'
        elif col == 'hba1c':
            col_formatted = 'HbA1c'
        else:
            col_formatted = col.replace('_', r'\_')
        # Can change to multicolumn{1}{c}{col_formatted} to center the colnames
        header += ' & ' + r'\multicolumn{1}{c}{' + col_formatted + '}'
        # header += ' & ' + col_formatted
    header += r' \\'
    latex_lines.append(header)
    latex_lines.append(r'\midrule')
    
    # Process rows with nested multirow
    current_k = None
    current_method = None
    
    for idx, row in df.iterrows():
        k_value = row['K']
        method = row['Method']
        tir_range = row['TIR']
        
        line = ''
        
        # Handle K value column (outer multirow)
        if k_value != current_k:
            current_k = k_value
            # Count how many rows have this K value
            rows_in_k = len(df[df['K'] == k_value]) + 0.6  # +0.6 to account for the midrule
            line += r'\multirow{' + str(rows_in_k) + r'}{*}{' + k_value + r'}'
        else:
            line += ''
        
        # Handle Method column (inner multirow)
        if method != current_method:
            current_method = method
            # Count how many rows have this method within the current K
            rows_in_method = len(df[(df['K'] == k_value) & (df['Method'] == method)])
            if method:  # Only add multirow if method is not empty
                line += ' & ' + r'\multirow{' + str(rows_in_method) + r'}{*}{' + method + r'}'
            else:
                line += ' & '
        else:
            line += ' & '
        
        # Format TIR range nicely
        tir_formatted = tir_range.replace('TIR_', 'TIR ').replace('_', '--')
        line += ' & ' + tir_formatted
        
        # Add the correlation values
        for col in col_interest:
            line += ' & ' + str(row[col])
        
        line += r' \\'
        latex_lines.append(line)
        
        # Add midrule after each method group (except within the same K)
        if idx < len(df) - 1:
            next_k = df.iloc[idx + 1]['K']
            next_method = df.iloc[idx + 1]['Method']
            # Add midrule when changing method (but not K) or when changing K
            if next_k != k_value:
                latex_lines.append(r'\midrule')
            elif next_method != method:
                latex_lines.append(r'\cmidrule(lr){2-' + str(3 + num_cols) + '}')
    
    # End table
    latex_lines.append(r'\bottomrule')
    latex_lines.append(r'\end{tabular}')
    latex_lines.append(r'\begin{tablenotes}')
    latex_lines.append(r'\scriptsize')
    latex_lines.append(r'\item $\qquad$ * $p<0.05$, † $p<0.01$, ‡ $p<0.001$')
    latex_lines.append(r'\end{tablenotes}')
    latex_lines.append(r'\end{table}')
    
    return '\n'.join(latex_lines)

# Generate the LaTeX table
latex_table = create_latex_table_with_nested_multirow(combined_kendall)

# Save to file
output_dir = os.path.join(cwd, 'results', 'tables')
output_file = os.path.join(output_dir, 'kendall_tau_table.tex')

with open(output_file, 'w') as f:
    f.write(latex_table)

# print(f"LaTeX table saved to: {output_file}")
print("\n" + "="*80)
print("LaTeX code:")
print("="*80)
print(latex_table)


LaTeX code:
\begin{table}[htbp]
\centering
\caption{Kendall's $\tau$ correlation between TIR metrics and health variables, with symbols indicating significant $p$-values under the null hypothesis of $\tau = 0$.}
\label{tab:kendall_tau}
\footnotesize
\begin{tabular}{lllrrrrr}
\toprule
$K$ & Method & Range (mg/dL) & \multicolumn{1}{c}{HDL-C} & \multicolumn{1}{c}{LDL-C} & \multicolumn{1}{c}{Total-C} & \multicolumn{1}{c}{TG} & \multicolumn{1}{c}{TG/HDL-C} \\
\midrule
\multirow{6.6}{*}{$K=2$} & \multirow{3}{*}{Traditional} & TIR 40--69 & 0.060$^*$ & 0.002$\:\,$ & -0.000$\:\,$ & -0.086$^\dagger$ & -0.091$^\dagger$ \\
 &  & TIR 70--180 & 0.012$\:\,$ & -0.005$\:\,$ & -0.015$\:\,$ & -0.024$\:\,$ & -0.021$\:\,$ \\
 &  & TIR 181--400 & -0.033$\:\,$ & -0.002$\:\,$ & 0.018$\:\,$ & 0.079$^\dagger$ & 0.071$^*$ \\
\cmidrule(lr){2-8}
 & \multirow{3}{*}{DE} & TIR 40--95 & 0.088$^\dagger$ & -0.013$\:\,$ & -0.017$\:\,$ & -0.113$^\ddagger$ & -0.119$^\ddagger$ \\
 &  & TIR 96--169 & -0.027$\:\,$ & -0.002$\

## Export Linear Model Results to LaTeX

In [ ]:
# Function to create LaTeX table for linear model results
def create_lm_results_latex_table(df):
    """
    Convert the lm_results dataframe to LaTeX booktabs format with multicolumn headers
    """
    latex_lines = []
    
    # Begin table
    latex_lines.append(r'\begin{table}[t]')
    latex_lines.append(r'\centering')
    latex_lines.append(r"\caption{Comparison of linear model fits with TIR compositional predictors based on consensus (CS) and data-driven (DE) thresholds. $\Delta$AIC denotes the difference AIC$_{\text{CS}}$ - AIC$_{\text{DE}}$. DE significantly outperforms CS in all cases, according to Clarke's observation-wise likelihood-ratio test for non-nested model comparison at  $\alpha = 0.05$.}")
    latex_lines.append(r'\label{tab:lm_results}')
    latex_lines.append(r'\small')
    
    # Count columns for K=2 and K=4
    k2_cols = [col for col in df.columns if '(K=2)' in col]
    k4_cols = [col for col in df.columns if '(K=4)' in col]
    
    # Column specification
    num_k2 = len(k2_cols)
    num_k4 = len(k4_cols)
    latex_lines.append(r'\begin{tabular}{l' + 'r' * (num_k2 + num_k4) + '}')
    latex_lines.append(r'\toprule')
    
    # Multi-column header for K=2 and K=4
    header_line1 = ''
    if num_k2 > 0:
        header_line1 += ' & ' + r'\multicolumn{' + str(num_k2) + r'}{c}{$K=2$}'
    if num_k4 > 0:
        header_line1 += ' & ' + r'\multicolumn{' + str(num_k4) + r'}{c}{$K=4$}'
    header_line1 += r' \\'
    latex_lines.append(header_line1)
    
    # Sub-header with actual column names
    header_line2 = '\cmidrule(lr){2-4} \cmidrule(lr){5-7} Response'
    # Extract base column names (without K=2 or K=4 suffix)
    for col in k2_cols:
        base_name = col.replace(' (K=2)', '')
        # Format column name
        if 'R^2 (data-driven)' in base_name:
            formatted = r'$R^2_{\text{DE}}$'
        elif 'R^2 (traditional)' in base_name:
            formatted = r'$R^2_{\text{CS}}$'
        elif 'AIC' in base_name:
            formatted = r'$\Delta$AIC'
        else:
            formatted = base_name
        header_line2 += ' & ' + r'\multicolumn{1}{c}{' + formatted + '}'
    
    for col in k4_cols:
        base_name = col.replace(' (K=4)', '')
        # Format column name
        if 'R^2 (data-driven)' in base_name:
            formatted = r'$R^2_{\text{DE}}$'
        elif 'R^2 (traditional)' in base_name:
            formatted = r'$R^2_{\text{CS}}$'
        elif 'AIC' in base_name:
            formatted = r'$\Delta$AIC'
        else:
            formatted = base_name
        header_line2 += ' & ' + r'\multicolumn{1}{c}{' + formatted + '}'
    
    header_line2 += r' \\'
    latex_lines.append(header_line2)
    latex_lines.append(r'\midrule')
    
    # Data rows
    for idx in df.index:
        # Format response variable name
        if idx == 'log_triglycerides':
            response_name = 'TG'
        elif idx == 'AIP':
            response_name = 'TG/HDL-C'
        elif idx == 'hdl_c':
            response_name = 'HDL-C'
        elif idx == 'ldl_c':
            response_name = 'LDL-C'
        elif idx == 'total_c':
            response_name = 'Total-C'
        elif idx == 'hba1c':
            response_name = 'HbA1c'
        else:
            response_name = idx.replace('_', r'\_')
        
        line = response_name
        
        # Add K=2 values
        for col in k2_cols:
            value = df.loc[idx, col]
            line += ' & ' + str(value)
        
        # Add K=4 values
        for col in k4_cols:
            value = df.loc[idx, col]
            line += ' & ' + str(value)
        
        line += r' \\'
        latex_lines.append(line)
    
    # End table
    latex_lines.append(r'\bottomrule')
    latex_lines.append(r'\end{tabular}')
    latex_lines.append(r'\begin{tablenotes}')
    latex_lines.append(r'\footnotesize')
    latex_lines.append(r'\item $\qquad$ * $p<0.05$, † $p<0.01$, ‡ $p<0.001$')
    latex_lines.append(r'\end{tablenotes}')
    latex_lines.append(r'\end{table}')
    
    return '\n'.join(latex_lines)

# Generate the LaTeX table
lm_latex_table = create_lm_results_latex_table(lm_results)

# Save to file
output_dir = os.path.join(cwd, 'results', 'tables')
output_file_lm = os.path.join(output_dir, 'lm_results_table.tex')

with open(output_file_lm, 'w') as f:
    f.write(lm_latex_table)

print(f"LaTeX table saved to: {output_file_lm}")
print("\n" + "="*80)
print("LaTeX code:")
print("="*80)
print(lm_latex_table)

LaTeX table saved to: g:\My Drive\Experiments\OptiThresholds\results\tables\lm_results_table.tex

LaTeX code:
\begin{table}[htbp]
\centering
\caption{Comparison of linear model fits with TIR compositional predictors based on consensus (CS) and data-driven (DE) thresholds. $\Delta$AIC denotes the difference AIC$_{\text{CS}}$ - AIC$_{\text{DE}}$. DE significantly outperforms CS in all cases, according to Clarke's observation-wise likelihood-ratio test for non-nested model comparison at  $\alpha = 0.05$.}
\label{tab:lm_results}
\small
\begin{tabular}{lrrrrrr}
\toprule
 & \multicolumn{3}{c}{$K=2$} & \multicolumn{3}{c}{$K=4$} \\
\cmidrule(lr){2-4} \cmidrule(lr){5-7} Response & \multicolumn{1}{c}{$R^2_{\text{DE}}$} & \multicolumn{1}{c}{$R^2_{\text{Trad}}$} & \multicolumn{1}{c}{$\Delta$AIC} & \multicolumn{1}{c}{$R^2_{\text{DE}}$} & \multicolumn{1}{c}{$R^2_{\text{Trad}}$} & \multicolumn{1}{c}{$\Delta$AIC} \\
\midrule
HDL-C & 0.018 & 0.011 & 4.2$^\ddagger$ & 0.041 & 0.014 & 16.2$^\ddagger$ \\
T